In [ ]:
import os
import dotenv

dotenv.load_dotenv()

if "DATA_PATH" not in os.environ:
    
    raise Exception("Please set the DATA_PATH environment variable to where you want to save your dataset")

DATA_DIR= os.environ["DATA_PATH"]
assert DATA_DIR, "Please set the DATA_PATH environment variable to your data directory"

# OUTPUT_DIR = os.path.join(DATA_DIR, "images/vsr") 
IMAGES_DIR = os.path.join(DATA_DIR, "images/vsr")
SAVE_DIR = os.path.join(DATA_DIR, "vsr")

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(IMAGES_DIR, exist_ok=True)


In [ ]:
from datasets import load_dataset


vsr_dataset = load_dataset("cambridgeltl/vsr_zeroshot")

In [ ]:
vsr_dataset['train'][0]

In [ ]:
import os
import requests
from datasets import load_dataset, Dataset
from tqdm import tqdm


In [ ]:
# download image and return local path
def download_and_replace_image(example):
    image_url = example["image_link"]
    image_name = example["image"]
    local_path = os.path.join(IMAGES_DIR, image_name)
    
    # Download only if not already present
    if not os.path.exists(local_path):
        try:
            response = requests.get(image_url, timeout=10)
            response.raise_for_status()
            with open(local_path, "wb") as f:
                f.write(response.content)
        except Exception as e:
            print(f"Failed to download {image_url}: {e}")
            local_path = None  # or set to empty string if needed

    # Replace the 'image' field with the local file path
    return {
        "image_path": local_path,
        "caption": example["caption"],
        "label": example["label"],
        "relation": example["relation"],
        "subj": example["subj"],
        "obj": example["obj"]
    }

In [ ]:
# Map the dataset with image downloading
vsr_dataset = vsr_dataset.map(download_and_replace_image, num_proc=8)

In [ ]:
# Remove unused columns (optional, in case you want a clean dataset)
vsr_dataset = vsr_dataset.remove_columns([col for col in vsr_dataset.column_names['train'] if col not in ["image_path", "caption", "label", "relation", "subj", "obj"]])

In [ ]:
# Sample
vsr_dataset['train'][0]

In [ ]:
# Save the dataset to disk
vsr_dataset.save_to_disk(SAVE_DIR) # # change this to your path where you want to save the dataset